# PHASE 4: Feature Engineering & Labeling
**Traceability**
- Issue ID: #4 Feature Engineering & Labeling

## 1. Objectives
- Engineer the Remaining Useful Life (RUL) target for both training and testing.
- Create classification labels for binary and multi-class failure risk.
- Derive temporal features using rolling windows to capture degradation dynamics.
- Normalize cycle counts to represent engine "age" as a relative fraction.

### 4.1 Import Libraries & Configure Engineering Parameters
We set thresholds for RUL clipping and classification windows based on domain knowledge.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Plotting Config ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
COLORS = ['#1F4E79', '#2E75B6', '#70AD47', '#FF7043', '#AB47BC']

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
DATA_DIR = Path('../data')
CLIP_VALUE = 125  # Optimum based on FD001 literature
WINDOW = 10       # Standard window size for rolling features
W1 = 30           # Warning threshold (binary class)
W0 = 15           # Critical threshold (multi-class)

### 4.2 RUL Label Engineering
Compute the raw RUL and apply piecewise clipping. For test data, we backfill RUL from the provided end-of-life ground truth.

In [ ]:
def add_rul_labels(df_train, df_test):
    """Compute and clip RUL for training and test data."""
    # 1. Train RUL
    rul_df = df_train.groupby('unit_number')['time_cycles'].max().reset_index()
    rul_df.columns = ['unit_number', 'max_cycle']
    df_train = df_train.merge(rul_df, on='unit_number')
    df_train['RUL_raw'] = df_train['max_cycle'] - df_train['time_cycles']
    df_train['RUL'] = df_train['RUL_raw'].clip(upper=CLIP_VALUE)
    df_train.drop(columns='max_cycle', inplace=True)
    
    # 2. Test RUL (backfilled from ground truth)
    y_test_rul = pd.read_csv(DATA_DIR / 'RUL_FD001.txt', sep=r'\s+', header=None, index_col=False, names=['RUL'])
    last_cycle = df_test.groupby('unit_number')['time_cycles'].max().reset_index()
    last_cycle.columns = ['unit_number', 'last_cycle']
    last_cycle['rul_at_end'] = y_test_rul['RUL'].values
    
    df_test = df_test.merge(last_cycle, on='unit_number')
    df_test['RUL_raw'] = df_test['rul_at_end'] + (df_test['last_cycle'] - df_test['time_cycles'])
    df_test['RUL'] = df_test['RUL_raw'].clip(upper=CLIP_VALUE)
    df_test.drop(columns=['last_cycle', 'rul_at_end'], inplace=True)
    
    return df_train, df_test

In [ ]:
def visualize_piecewise_rul(df, unit=1):
    """Visualize raw vs. clipped RUL for a specific engine."""
    subset = df[df['unit_number'] == unit]
    
    plt.figure(figsize=(10, 5))
    plt.plot(subset['time_cycles'], subset['RUL_raw'], label='Raw RUL', color=COLORS[3], linestyle='--')
    plt.plot(subset['time_cycles'], subset['RUL'], label=f'Piecewise RUL (Clip={CLIP_VALUE})', color=COLORS[0], linewidth=2)
    plt.axhline(y=CLIP_VALUE, color='gray', linestyle=':', alpha=0.5)
    plt.title(f'Piecewise RUL Labeling (Engine {unit})')
    plt.xlabel('Time Cycles')
    plt.ylabel('RUL')
    plt.legend()
    plt.show()

# Note: This needs to be run after add_rul_labels
# visualize_piecewise_rul(df_train, unit=1)

### 4.3 Classification Label Engineering
Create binary labels (failure within 30 cycles) and multi-class labels (Healthy, Warning, Critical).

In [ ]:
def visualize_rolling_effect(df, sensor='s_11', unit=1):
    """Visualize raw sensor vs. rolling mean."""
    subset = df[df['unit_number'] == unit]
    
    plt.figure(figsize=(12, 5))
    plt.plot(subset['time_cycles'], subset[sensor], label='Raw Sensor', color='gray', alpha=0.4)
    plt.plot(subset['time_cycles'], subset[f'{sensor}_rm'], label=f'Rolling Mean (w={WINDOW})', color=COLORS[1], linewidth=2)
    plt.title(f'Rolling Window Smoothing: {sensor} (Engine {unit})')
    plt.xlabel('Time Cycles')
    plt.ylabel('Value')
    plt.legend()
    plt.show()

# Note: This needs to be run after add_rolling_features
# visualize_rolling_effect(df_train, sensor='s_11', unit=1)

In [ ]:
def add_classification_labels(df):
    """Add binary and 3-class classification labels."""
    # label1: Binary (failure within 30 cycles)
    df['label1'] = (df['RUL_raw'] <= W1).astype(int)
    
    # label2: 3-class (0: healthy, 1: warning, 2: critical)
    df['label2'] = df['label1'].copy()
    df.loc[df['RUL_raw'] <= W0, 'label2'] = 2
    
    return df

### 4.4 Temporal & Domain Feature Engineering
Generate rolling window statistics per sensor and normalize the engine lifecycle cycle counts.

In [ ]:
def add_rolling_features(df, sensors, window=10):
    """Add rolling stats per sensor column (leakage-safe)."""
    df = df.copy().sort_values(['unit_number', 'time_cycles'])
    for col in sensors:
        grouped = df.groupby('unit_number')[col]
        df[f'{col}_rm']   = grouped.transform(lambda x: x.rolling(window, min_periods=1).mean())
        df[f'{col}_rstd'] = grouped.transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        df[f'{col}_rmin'] = grouped.transform(lambda x: x.rolling(window, min_periods=1).min())
        df[f'{col}_rmax'] = grouped.transform(lambda x: x.rolling(window, min_periods=1).max())
    return df

def add_domain_features(df):
    """Add cycle normalization and sensor differencing."""
    df = df.copy()
    # Cycle normalization
    max_c = df.groupby('unit_number')['time_cycles'].transform('max')
    df['cycle_norm'] = df['time_cycles'] / max_c
    
    # Rate of change for key sensors
    for col in ['s_9', 's_11', 's_14']:
        if col in df.columns:
            df[f'{col}_diff'] = df.groupby('unit_number')[col].transform(lambda x: x.diff().fillna(0))
    return df

### 4.5 Execution: Feature Pipeline
Apply all engineering steps to the cleaned datasets and save the final labeled features.

In [ ]:
# 1. Load Data
df_train = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test_cleaned.csv')
sensor_cols = [c for c in df_train.columns if c.startswith('s_')]
print(f"✅ Data loaded: Train {df_train.shape}, Test {df_test.shape}")

# 2. RUL Labeling
df_train, df_test = add_rul_labels(df_train, df_test)
print(f"✅ RUL labels engineered (Clipped at {CLIP_VALUE}).")
visualize_piecewise_rul(df_train, unit=1)

# 3. Classification Labels
df_train = add_classification_labels(df_train)
df_test = add_classification_labels(df_test)
print(f"✅ Classification labels added.")

# 4. Rolling Features
df_train = add_rolling_features(df_train, sensor_cols, window=WINDOW)
df_test = add_rolling_features(df_test, sensor_cols, window=WINDOW)
print(f"✅ Rolling window features added.")
visualize_rolling_effect(df_train, sensor='s_11', unit=1)

# 5. Domain Features
df_train = add_domain_features(df_train)
df_test = add_domain_features(df_test)
print(f"✅ Domain features added.")

# 6. Save Labeled Data
df_train.to_csv(PROCESSED_DIR / 'train_labeled.csv', index=False)
df_test.to_csv(PROCESSED_DIR / 'test_labeled.csv', index=False)
print(f"\n✅ Labeled & engineered data saved to {PROCESSED_DIR}")